# Search FeatureSet for Survival

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
import os
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, roc_curve,
)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

import shap
import warnings
warnings.filterwarnings('ignore')

from scipy.stats import t as student_t

import matplotlib.pyplot as plt
import itertools
from itertools import combinations
from tqdm import tqdm


METRIC_NAMES = ["Accuracy", "Precision", "Recall", "F1-score", "AUROC", "AUPRC"]
MODEL_COLORS_ROC = ['#81989B', '#D19246', '#B5AF8B', '#7EA4B6', '#71A682', '#86BC79', '#B8DBB3', '#4A4F7E']
METRIC_COLORS = ['#6AD1A3', '#7FBDDA', '#BBC7BE','#FFD47D', '#FFA288', '#C49892']
MODEL_COLORS_BAR = ['#6AD1A3', '#7FBDDA', '#BBC7BE', '#FFD47D', '#FFA288', '#C49892', '#84ADDC']

DATASET_PATH='../../../datasets/train_set.csv'
SHAP_KERNEL_BG = 60
SHAP_KERNEL_VAL = 120
SHAP_LINEAR_BG = 200
SAVE_DIR = "../../../results/survival"
TARGET = "Survival"
os.makedirs(SAVE_DIR, exist_ok=True)

/root/miniconda3/envs/starr/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# prepare

In [ ]:
# -----------------------------
# feature selection
# -----------------------------
def feature_selection(df, target):

    # Medication features (fixed as input, must be included for non-resistance tasks)
    polyType_cols = ['colistin_cms_daily_freq', 'polymyxin_b_daily_freq', 'colistin_sulfate_daily_freq']
    Combination_medication_cols = [
        'carbapenem_daily_dose', 'sulbactam_daily_dose', 'tigecycline_daily_dose',
        'minocycline_daily_dose', 'vancomycin_daily_dose', 'eravacycline_daily_dose',
        'aminoglycoside_daily_dose'
    ]
    medication_features = polyType_cols + Combination_medication_cols

    time_features = ['Pre_Hospital_Days', 'Pre_ICU_Days']
    base_cols = ['Age', 'Gender', 'BMI']

    comorb_cols = ['Diabetes Mellitus', 'Hypertension', 'Heart Disease', 'Stroke', 'Malignant Tumor',
        'Chronic Kidney Disease', 'Chronic Liver Disease', 'COPD', 'Comorb_other']

    df = df.copy()
    df[comorb_cols] = df[comorb_cols].fillna(0)
    df['Comorb_count'] = df[comorb_cols].sum(axis=1)
    comorb_cols = comorb_cols + ['Comorb_count']

    immuno_cols = ['Use immunosuppressive agents', 'Neutrophil Reduction', 'HIV/AIDS',
        'Post-Transplant Status', 'Chemotherapy/Radiation', 'immuno_Other']

    support_cols = ['Resp_support', 'Oxygen_concentration']

    pre_lab_cols = ['WBC', 'N_percent', 'L_count', 'PLT', 'CRP1', 'PCT1', 'D-d', 'Cr_baseline', 'eGFR1', 'RRT', 'ALT', 'AST', 'TB', 'ALB']
    dynamic_lab_cols=[]
    lab_cols = pre_lab_cols + dynamic_lab_cols

    infection_cols = ['Infection_HAP', 'Infection_VAP']
    Coinfection_cols = ['Coinfection_G_Pos', 'Coinfection_G_Neg', 'Coinfection_Fungi']
    df[Coinfection_cols] = df[Coinfection_cols].fillna(0).astype(int)
    infection_cols = infection_cols + Coinfection_cols

    # ===== RSI mapping (R/I=1, S=0) =====
    resistance_features = ['resistance_SXT', 'resistance_KAN', 'resistance_MIN', 'resistance_TGC', 'resistance_CFP-SUL', 'resistance_TOB']
    mapping = {'R': 1, 'I': 1, 'S': 0}

    for c in resistance_features:
        if c in df.columns:
            s = df[c].astype(str).str.strip().str.upper()
            df[c] = pd.to_numeric(s.map(mapping), errors="coerce")

    # Only these two groups are collapsed in SHAP ranking / group enumeration
    group_defs = {
        "Comorbidity": [c for c in comorb_cols if c in df.columns],
        "Immunosuppression": [c for c in immuno_cols if c in df.columns],
    }

    if target == 'Target_Polymyxin':
        feature_cols = base_cols + time_features + comorb_cols + immuno_cols + support_cols + pre_lab_cols
        include_med = False
    else:
        feature_cols = (
            base_cols + comorb_cols + immuno_cols + support_cols +
            lab_cols + infection_cols + resistance_features +
            polyType_cols + Combination_medication_cols
        )
        include_med = True

    feature_cols = [c for c in feature_cols if c in df.columns]

    return feature_cols, df, medication_features, group_defs, include_med

In [3]:
# -----------------------------
# Models
# -----------------------------
def get_models(spw):
    models={
        'XGBoost': XGBClassifier(scale_pos_weight=spw, use_label_encoder=False, objective='binary:logistic', eval_metric='logloss', verbosity=0, random_state=42, n_jobs=1),
        'LightGBM': LGBMClassifier(class_weight='balanced', verbosity=-1, random_state=42, n_jobs=1),
        'CatBoost': CatBoostClassifier(auto_class_weights='Balanced', verbose=0, loss_function='Logloss', allow_writing_files=False, random_state=42, thread_count=1),
        'RandomForest': RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=1),
        'LogisticRegression': LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000),
        'SVM': SVC(class_weight='balanced',random_state=42, probability=True),
        'KNN': KNeighborsClassifier(weights='distance', n_jobs=1),
    }
    return models

In [4]:
# -----------------------------
# Pipeline utilities
# -----------------------------
TREE_MODELS = {"XGBoost", "LightGBM", "CatBoost", "RandomForest"}
def is_tree(model_name): 
    return model_name in TREE_MODELS


def create_pipeline(model, model_name, use_smote=False, smote_ratio=0.8):
    steps = [("imputer", SimpleImputer(strategy="median"))]

    if not is_tree(model_name):
        steps.append(("scaler", StandardScaler()))

    if use_smote:
        steps.append(("smote", SMOTE(random_state=42, sampling_strategy=smote_ratio)))

    steps.append(("clf", clone(model)))

    if use_smote:
        return ImbPipeline(steps)
    return Pipeline(steps)


def get_preprocess_transformer(pipe: Pipeline):
    if "clf" not in pipe.named_steps:
        raise ValueError("Pipeline must have a 'clf' step.")
    return pipe[:-1]

In [ ]:
# -----------------------------
# Metrics / CV report
# -----------------------------
def evaluate_model(y_true, y_pred, y_proba):
    return {
        "Accuracy": float(accuracy_score(y_true, y_pred)),
        "Precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "Recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "F1-score": float(f1_score(y_true, y_pred, zero_division=0)),
        "AUROC": float(roc_auc_score(y_true, y_proba)),
        "AUPRC": float(average_precision_score(y_true, y_proba)),
    }

def cv_metrics(model, model_name, X, y, cv=5, use_smote=False, threshold=0.5, random_state=42, return_ci=False, n_fpr=200):

    kfold = StratifiedKFold(n_splits=cv, shuffle=True, random_state=random_state)
    per_fold = {k: [] for k in METRIC_NAMES}
    tprs, aucs = [], []
    mean_fpr = np.linspace(0, 1, n_fpr) if return_ci else None

    for tr, va in kfold.split(X, y):
        X_tr, X_va = X.iloc[tr], X.iloc[va]
        y_tr, y_va = y.iloc[tr].values, y.iloc[va].values

        pipe = create_pipeline(model if return_ci else clone(model), model_name, use_smote=use_smote)
        pipe.fit(X_tr, y_tr)

        proba = pipe.predict_proba(X_va)[:, 1]
        pred = (proba >= threshold).astype(int)

        metrics = evaluate_model(y_va, pred, proba)
        for k in METRIC_NAMES:
            per_fold[k].append(float(metrics[k]))

        if return_ci:
            fpr, tpr, _ = roc_curve(y_va, proba)
            auc = roc_auc_score(y_va, proba)
            tpr_i = np.interp(mean_fpr, fpr, tpr)
            tpr_i[0] = 0.0
            tprs.append(tpr_i)
            aucs.append(float(auc))

    # calculate 6 metrics mean
    metrics_mean = {k: float(np.mean(per_fold[k])) for k in METRIC_NAMES}

    if not return_ci:
        return metrics_mean

    # calculate CI
    n = cv
    dof = n - 1
    t_val = student_t.ppf(0.975, dof) if n > 1 else 0.0

    mean_tpr = np.mean(tprs, axis=0)
    mean_tpr[-1] = 1.0
    std_tpr = np.std(tprs, axis=0, ddof=1) if n > 1 else np.zeros_like(mean_tpr)
    tpr_upper = np.clip(mean_tpr + t_val * std_tpr / np.sqrt(n), 0, 1)
    tpr_lower = np.clip(mean_tpr - t_val * std_tpr / np.sqrt(n), 0, 1)

    mean_auc = float(np.mean(aucs))
    std_auc = float(np.std(aucs, ddof=1)) if n > 1 else 0.0
    auc_ci = (mean_auc - t_val * std_auc / np.sqrt(n), mean_auc + t_val * std_auc / np.sqrt(n))

    metrics_ci = {}
    for k in METRIC_NAMES:
        vals = np.array(per_fold[k], dtype=float)
        s = float(np.std(vals, ddof=1)) if n > 1 else 0.0
        m = metrics_mean[k]
        metrics_ci[k] = (m - t_val * s / np.sqrt(n), m + t_val * s / np.sqrt(n))

    return {
        "mean_fpr": mean_fpr,
        "mean_tpr": mean_tpr,
        "tpr_upper": tpr_upper,
        "tpr_lower": tpr_lower,
        "aucs": aucs,
        "mean_auc": mean_auc,
        "std_auc": std_auc,
        "auc_ci": auc_ci,
        "metrics_per_fold": per_fold,
        "metrics_mean": metrics_mean,
        "metrics_ci": metrics_ci,
    }

# SHAP

In [ ]:
# -----------------------------
# CV-SHAP for all models
# -----------------------------
def _sample_indices(n, k, seed):
    if k is None or k <= 0 or k >= n:
        return np.arange(n)
    rng = np.random.RandomState(seed)
    return rng.choice(n, size=k, replace=False)

def _ensure_shap_2d(shap_vals):
    if isinstance(shap_vals, list):
        shap_vals = shap_vals[1]
    shap_vals = np.asarray(shap_vals)
    if shap_vals.ndim == 3 and shap_vals.shape[2] == 2:
        shap_vals = shap_vals[:, :, 1]
    return shap_vals


def cv_shap_importance(models, X, y, cv=5, random_state=42,
    medication_features=None, group_defs=None, exclude_medication_in_ranking=True,
    use_smote=False, verbose=True, return_oof_shap=True,
    kernel_background_samples=SHAP_KERNEL_BG, kernel_val_samples=SHAP_KERNEL_VAL, linear_background_samples=SHAP_LINEAR_BG):

    assert isinstance(X, pd.DataFrame), "X must be a pandas DataFrame"
    y = pd.Series(y).astype(int)

    medication_features = medication_features or []
    group_defs = group_defs or {"Comorbidity": [], "Immunosuppression": []}

    # map feature -> group (only two groups)
    feat_to_group = {}
    for g, cols in group_defs.items():
        for c in cols:
            if c in X.columns:
                feat_to_group[c] = g

    kfold = StratifiedKFold(n_splits=cv, shuffle=True, random_state=random_state)
    results = {}

    for model_name, model in models.items():
        if verbose:
            print(f"\n===== CV-SHAP (pipeline): {model_name} =====")

        fold_imps = []
        list_shap = []
        list_Xt = []

        for fold, (train, val) in enumerate(kfold.split(X, y), start=1):
            X_train, X_val = X.iloc[train], X.iloc[val]
            y_train = y.iloc[train].values

            pipe = create_pipeline(model, model_name, use_smote=use_smote)
            pipe.fit(X_train, y_train)

            preprocess = get_preprocess_transformer(pipe)
            clf = pipe.named_steps["clf"]

            # Transform to model feature space (numeric array)
            Xt_train = preprocess.transform(X_train)
            Xt_val = preprocess.transform(X_val)

            # choose explainer by model type
            if is_tree(model_name):
                explainer = shap.TreeExplainer(clf)
                shap_vals = explainer.shap_values(Xt_val)
                shap_vals = _ensure_shap_2d(shap_vals)
                Xt_used = Xt_val

            elif model_name == "LogisticRegression":
                # background in transformed space
                idx_bg = _sample_indices(Xt_train.shape[0], linear_background_samples, seed=random_state + fold)
                Xt_bg = Xt_train[idx_bg]
                explainer = shap.LinearExplainer(clf, Xt_bg, feature_perturbation="interventional")
                shap_vals = explainer.shap_values(Xt_val)
                shap_vals = _ensure_shap_2d(shap_vals)
                Xt_used = Xt_val

            else:
                # KernelExplainer for SVM/KNN (slow): subsample background and validation
                idx_bg = _sample_indices(Xt_train.shape[0], kernel_background_samples, seed=random_state + fold)
                Xt_bg = Xt_train[idx_bg]

                idx_va = _sample_indices(Xt_val.shape[0], kernel_val_samples, seed=random_state + 1000 + fold)
                Xt_va_use = Xt_val[idx_va]

                # function expects transformed features
                def f(z):
                    z = np.asarray(z)
                    if hasattr(clf, "predict_proba"):
                        return clf.predict_proba(z)[:, 1]
                    if hasattr(clf, "decision_function"):
                        s = clf.decision_function(z)
                        return 1.0 / (1.0 + np.exp(-s))
                    raise ValueError(f"{clf.__class__.__name__} lacks predict_proba and decision_function.")

                explainer = shap.KernelExplainer(f, Xt_bg)
                shap_vals = explainer.shap_values(Xt_va_use, nsamples="auto")
                shap_vals = _ensure_shap_2d(shap_vals)
                Xt_used = Xt_va_use

            # align with original feature columns (one-hot already, preprocess doesn't change feature count)
            if shap_vals.ndim != 2 or shap_vals.shape[1] != X.shape[1]:
                raise ValueError(
                    f"{model_name} fold {fold}: SHAP shape {shap_vals.shape} != (n, {X.shape[1]}). "
                    "Make sure X is already one-hot numeric and pipeline doesn't change feature count."
                )
            if return_oof_shap:
                list_shap.append(shap_vals)
                list_Xt.append(Xt_used)
            mean_abs = np.abs(shap_vals).mean(axis=0)
            fold_imps.append({feat: float(mean_abs[i]) for i, feat in enumerate(X.columns)})

            if verbose:
                print(f"  fold {fold}: done (val_n={len(X_val)})")

        # average across folds
        avg_imp = {feat: float(np.mean([d.get(feat, 0.0) for d in fold_imps])) for feat in X.columns}
        raw_df = (
            pd.DataFrame({"feature": list(avg_imp.keys()), "mean_abs_shap": list(avg_imp.values())})
            .sort_values("mean_abs_shap", ascending=False)
            .reset_index(drop=True)
        )

        if exclude_medication_in_ranking and medication_features:
            raw_rank_df = raw_df[~raw_df["feature"].isin(medication_features)].reset_index(drop=True)
        else:
            raw_rank_df = raw_df.copy()

        # group aggregation (only two groups collapsed)
        group_scores = {}
        
        for _, row in raw_rank_df.iterrows():
            feat = str(row["feature"])
            val = float(row["mean_abs_shap"])
            g = feat_to_group.get(feat, feat)
            group_scores[g] = group_scores.get(g, 0.0) + val

        grouped_df = (
            pd.DataFrame({"group": list(group_scores.keys()), "mean_abs_shap": list(group_scores.values())})
            .sort_values("mean_abs_shap", ascending=False)
            .reset_index(drop=True)
        )

        results[model_name] = {
            "feature_importance": raw_df,
            "feature_importance_grouped": grouped_df,
        }
        if return_oof_shap and list_shap:
            results[model_name]["shap_oof"] = np.vstack(list_shap)
            results[model_name]["Xt_oof"] = np.vstack(list_Xt)
            results[model_name]["feature_names"] = X.columns.tolist()

    return results

# Feature-set selection

## Stage A
- per model: search top sets(top10 feature->top7 feature set)
- get 7models x 7sets

In [8]:
# -----------------------------
# 1) group_defs -> raw columns
# -----------------------------
def build_group_to_cols(feature_cols, group_defs):
    """
    feature_cols: List[str]             raw columns used in training (already one-hot / imputer + scaler)
    return: group_to_cols: Dict[str, List[str]]    eg. {"Comorbidity":[...], "Immunosuppression":[...]}
    """
    group_to_cols = {g: cols[:] for g, cols in group_defs.items()}
    grouped_cols = set(sum(group_defs.values(), []))  # all columns already assigned to some group

    for c in feature_cols:
        if c in grouped_cols:
            continue
        group_to_cols.setdefault(c, [c])  # singleton group

    return group_to_cols


def groups_to_feature_cols(groups, group_to_cols, medication_features = None, include_med = False):
    """
    groups -> expanded raw columns (dedup, keep order).
    Optionally prepend medication_features when include_med=True.
    """
    cols = []
    for g in groups:
        cols.extend(group_to_cols.get(g, [g]))

    # de-dup keep order
    cols = list(dict.fromkeys(cols))

    if include_med and medication_features:
        cols = list(dict.fromkeys(medication_features + cols))

    return cols

In [9]:
# -----------------------------
# 2) enumerate group subsets
# -----------------------------
def enumerate_group_subsets(top_groups, max_k=10):
    n = len(top_groups)
    top_groups = top_groups[:max_k]
    for k in range(1, min(max_k, n) + 1):
        for comb in combinations(top_groups, k):
            yield list(comb)

In [10]:
# -----------------------------
# 3)for each model: search top sets by AUROC(top10 feature->top7 feature-set)
# ----------------------------- 
def search_topsets_per_model(model, model_name, X_train, y_train, top10_groups, group_to_cols, medication_features, include_med, cv=5, use_smote=False, top_sets=7, max_k=10):
    """
    to per model: enumerate all subsets of topk_groups, select top_sets sorted by auroc
    return: List[(groups, score, ncols)]
    """
    results = []
    from math import comb
    n = min(len(top10_groups), max_k)
    actual_total = sum(comb(n, k) for k in range(1, n + 1)) if n > 0 else 0
    for groups in tqdm(enumerate_group_subsets(top10_groups, max_k=max_k), desc=model_name, total=actual_total):        
        cols = groups_to_feature_cols(groups, group_to_cols, medication_features, include_med)
        # Score subset by AUROC
        score = cv_metrics(model, model_name, X=X_train[cols], y=y_train, cv=cv, use_smote=use_smote)["AUROC"]
        results.append((groups, score, len(cols)))

    results.sort(key=lambda x: x[1], reverse=True)
    return results[:top_sets]


In [11]:
# -----------------------------
# union and dedup candidates
# ----------------------------- 
def union_dedup_candidates(per_model_top_sets):
    """ merge all models' top_sets candidates, dedup by "candidates", get <= 49 candidates """
    seen = set()
    union = []
    for model_name, lst in per_model_top_sets.items():
        for candidates, score, ncols in lst:
            key = tuple(candidates)
            if key not in seen:
                seen.add(key)
                union.append({"candidates": candidates, "seed_score": score, "ncols": ncols})
    return union

In [12]:
# =========================================================
# 4) get top7 set per model
# =========================================================
def search_candidates(X_train, y_train, feature_cols, group_defs, models, shap_results, medication_features, include_med, cv=5, use_smote=False, top_sets=7, max_k=10, verbose=True):
    """
    Input:
      - feature_cols, group_defs:   from feature_selection (no need to recalculate)
      - models:                     model dictionary (no need to recalculate)
      - shap_results:               each model's top10_groups (you already have)
      - include_med:                (outcome/survival/duration=True; resistance=False)

    Output:
      - per_model_top_sets: each model's top7 candidate sets (groups-level)
      - group_to_cols: for later expand + full evaluation
    """
    # group -> raw cols
    group_to_cols = build_group_to_cols(feature_cols, group_defs)
    per_model_top_sets = {}

    for model_name, model in models.items():
        top10 = shap_results[model_name]["feature_importance_grouped"]["group"].head(10).tolist()
        per_model_top_sets[model_name] = search_topsets_per_model(model, model_name, X_train, y_train, top10[:max_k], group_to_cols, medication_features, include_med, cv, use_smote, top_sets, max_k)
        if verbose:
            print(f"model: {model_name}, top10: {top10}")
            for i in per_model_top_sets[model_name]:
                print(i)
    return per_model_top_sets, group_to_cols

## Stage B
For each candidate set, run the "full model × CV mean" calculation to obtain the final metric for the set (7-model average).

In [13]:
def evaluate_candidates(
    candidates, models, X_train, y_train, group_to_cols,
    medication_features, include_med,
    cv=5, use_smote=False, random_state=42, threshold=0.5
):
    rows = []
    candidate_details = {}

    for i, cand in enumerate(candidates, start=1):
        set_id = f"cand_{i:02d}"
        groups = cand["candidates"]
        cols = groups_to_feature_cols(groups, group_to_cols, medication_features, include_med)
        X_sub = X_train[cols]

        per_model_metrics = {}
        for model_name, model in models.items():
            per_model_metrics[model_name] = cv_metrics(model=model, model_name=model_name, X=X_sub, y=y_train, cv=cv, use_smote=use_smote, random_state=random_state, threshold=threshold,return_ci=False)

        # final metric: set's mean over models
        set_metric_mean = {}
        for k in METRIC_NAMES:
            set_metric_mean[k] = float(np.mean([per_model_metrics[m][k] for m in per_model_metrics]))

        candidate_details[set_id] = {"groups": groups, "cols": cols, "per_model_metrics": per_model_metrics}

        row = {"set_id": set_id, "groups": "|".join(groups), "n_groups": len(groups), "n_cols": len(cols)}
        for k in METRIC_NAMES:
            row[f"{k}_mean_over_models"] = set_metric_mean[k]
        rows.append(row)

    set_scores_df = pd.DataFrame(rows)
    set_scores_df = set_scores_df.sort_values("AUROC_mean_over_models", ascending=False).reset_index(drop=True)
    return set_scores_df, candidate_details

## stage C
Select the optimal set for each of the 6 metrics → Obtain the final 6 sets (deduplicated)

In [ ]:
import numpy as np

def select_final_sets(set_scores_df, candidate_details, drop_pct=0.4, min_models_kept=5, verbose=True):
    best_set_per_metric = {}

    model_names = list(next(iter(candidate_details.values()))["per_model_metrics"].keys())
    set_ids = list(candidate_details.keys())
    n_sets = len(set_ids)
    n_keep = max(1, int(n_sets * (1 - drop_pct)))

    for k in METRIC_NAMES:
        kept_by_model = {}
        for m in model_names:
            scores = {sid: float(candidate_details[sid]["per_model_metrics"][m][k]) for sid in set_ids}
            sorted_sets = sorted(set_ids, key=lambda s: scores[s], reverse=True)
            threshold_score = scores[sorted_sets[n_keep - 1]]
            kept_by_model[m] = set(sid for sid in set_ids if scores[sid] >= threshold_score)

        eligible = set(set_ids)
        for m in model_names:
            eligible = eligible & kept_by_model[m]
        eligible = list(eligible)

        if not eligible:
            eligible = [sid for sid in set_ids
                       if sum(1 for m in model_names if sid in kept_by_model[m]) >= min_models_kept]
            if verbose:
                print(f"[{k}] Intersection is empty, fallback to keeping at least {min_models_kept} models, eligible_n={len(eligible)}")

        col = f"{k}_mean_over_models"
        if eligible:
            sub = set_scores_df[set_scores_df["set_id"].isin(eligible)]
            best_set_id = sub.loc[sub[col].idxmax(), "set_id"]
            best_set_per_metric[k] = best_set_id
        else:
            best_set_id = set_scores_df.loc[set_scores_df[col].idxmax(), "set_id"]
            best_set_per_metric[k] = best_set_id
            if verbose:
                print(f"[{k}] No qualified set, fallback to the global highest mean {best_set_id}")

    return best_set_per_metric

## stage D
5-fold CV on the final feature-sets (no gridsearch)

In [15]:
def evaluate_final_sets(best_set_per_metric, candidate_details, models, X_train, y_train, cv=5, use_smote=False, random_state=42):
    final_set_eval = {}
    metrics_rows = []

    final_set_ids = list(dict.fromkeys(best_set_per_metric.values()))
    set_id_to_metrics = {}
    for metric, set_id in best_set_per_metric.items():
        set_id_to_metrics.setdefault(set_id, []).append(metric)
    
    for set_id in final_set_ids:
        selected_by = set_id_to_metrics[set_id]
        print(f"set_id: {set_id}, selected_by: {selected_by}")

        groups = candidate_details[set_id]["groups"]
        cols = candidate_details[set_id]["cols"]
        X_sub = X_train[cols]

        final_set_eval[set_id] = {"groups": groups, "cols": cols, "selected_by": selected_by, "models": {}}

        for model_name, model in models.items():
            print("="*30, model_name,"="*30,"\n")

            cv_ci = cv_metrics(model, model_name, X=X_sub, y=y_train, cv=cv, use_smote=use_smote, random_state=random_state,return_ci=True)

            final_set_eval[set_id]["models"][model_name] = {
                "cv_ci": cv_ci,
            }

            row = {
                "set_id": set_id, "model_name": model_name, "selected_by": "|".join(selected_by),
                **{k: cv_ci["metrics_mean"][k] for k in METRIC_NAMES},
                "AUROC_lower": cv_ci["metrics_ci"]["AUROC"][0],
                "AUROC_upper": cv_ci["metrics_ci"]["AUROC"][1],
            }
            for k in METRIC_NAMES:
                print(f"{k}: {row[k]:.4f} | ", end="")
            print("\n")
            metrics_rows.append(row)

    final_set_metrics = pd.DataFrame(metrics_rows)
    return final_set_eval, final_set_metrics

# main

### prepare data and models

In [17]:
# ===== 1. load data =====
train_df = pd.read_csv(DATASET_PATH, encoding='gbk')

target = TARGET

train_df = train_df[train_df[target].notna()].copy()
train_df[target] = train_df[target].astype(int)
print(train_df[target].value_counts())
print(f"\ndata shape: {train_df.shape}")

# ===== 2. feature selection =====
feature_cols, train_df, medication_features, group_defs, include_med = feature_selection(train_df, target)

X_train = train_df[feature_cols]
y_train = train_df[target]

# ===== 3. models =====
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
models = get_models(scale_pos_weight)

Survival
1    325
0    118
Name: count, dtype: int64

data shape: (443, 126)


### Train with all-features

In [18]:
per_model_metrics = {}
for model_name, model in models.items():
    per_model_metrics[model_name] = cv_metrics(model, model_name, X_train, y_train, cv=5, use_smote=False, threshold=0.5, random_state=42, return_ci=True, n_fpr=200)

print(f"Model:              Accuracy precision recall f1-score AUROC AUPRC")
for model_name, metrics in per_model_metrics.items():
    print(f"{model_name:18}", end=" ")
    for metric_name, metric_value in metrics['metrics_mean'].items():
        print(f" {metric_value:.4f}", end=" ")
    print()

Model:              Accuracy precision recall f1-score AUROC AUPRC
XGBoost             0.7381  0.8168  0.8308  0.8233  0.7377  0.8817 
LightGBM            0.7313  0.8050  0.8369  0.8205  0.7382  0.8828 
CatBoost            0.7561  0.8241  0.8492  0.8358  0.7632  0.8932 
RandomForest        0.7404  0.7478  0.9754  0.8464  0.7629  0.8945 
LogisticRegression  0.6885  0.8499  0.6985  0.7667  0.7250  0.8592 
SVM                 0.7562  0.7734  0.9446  0.8504  0.7543  0.8893 
KNN                 0.7157  0.7641  0.8862  0.8205  0.6453  0.8077 


### SHAP -> feature importance rank -> top10_groups per model

In [20]:
# ===== 4. SHAP ranking (pipeline version) =====
shap_results = cv_shap_importance(
        models=models, X=X_train, y=y_train, cv=5,
        medication_features=medication_features if include_med else [],
        group_defs=group_defs, exclude_medication_in_ranking=True,
        use_smote=False, verbose=False, return_oof_shap=True
    )

100%|██████████| 88/88 [00:41<00:00,  2.12it/s]


In [21]:
for model_name in models:
    top10 = shap_results[model_name]["feature_importance_grouped"]["group"].head(10).tolist()
    print(f"{model_name}: {top10}")

XGBoost: ['PLT', 'Comorbidity', 'eGFR1', 'Age', 'PCT1', 'ALB', 'CRP1', 'D-d', 'BMI', 'Cr_baseline']
LightGBM: ['PLT', 'Comorbidity', 'eGFR1', 'Age', 'PCT1', 'ALB', 'CRP1', 'D-d', 'Oxygen_concentration', 'WBC']
CatBoost: ['PLT', 'eGFR1', 'Comorbidity', 'PCT1', 'Age', 'ALB', 'N_percent', 'D-d', 'Oxygen_concentration', 'Cr_baseline']
RandomForest: ['PLT', 'Comorbidity', 'PCT1', 'eGFR1', 'Cr_baseline', 'N_percent', 'ALB', 'Age', 'L_count', 'D-d']
LogisticRegression: ['Comorbidity', 'PLT', 'Immunosuppression', 'Age', 'N_percent', 'resistance_TGC', 'Oxygen_concentration', 'Resp_support', 'AST', 'ALB']
SVM: ['Comorbidity', 'PLT', 'Age', 'N_percent', 'resistance_TGC', 'eGFR1', 'CRP1', 'Oxygen_concentration', 'Immunosuppression', 'Cr_baseline']
KNN: ['Comorbidity', 'Immunosuppression', 'PLT', 'N_percent', 'Cr_baseline', 'eGFR1', 'resistance_MIN', 'Coinfection_Fungi', 'Age', 'CRP1']


In [ ]:
shap_dir=SAVE_DIR+'/shap_csv'
os.makedirs(shap_dir, exist_ok=True)

# 1. Medication Included
all_df = pd.concat([res["feature_importance"].assign(model=mn) for mn, res in shap_results.items()], ignore_index=True)
all_df.to_csv(shap_dir + "/importance_with_medication_no_group-nogridsearch.csv", index=False)

# 2. Medication Excluded
all_df = pd.concat([
        res["feature_importance"][~res["feature_importance"]["feature"].isin(medication_features)].assign(model=mn)
        for mn, res in shap_results.items()
    ], ignore_index=True)
all_df.to_csv(shap_dir + "/importance_no_medication_no_group-nogridsearch.csv", index=False)

# 3. Grouped
all_df = pd.concat([res["feature_importance_grouped"].assign(model=mn) for mn, res in shap_results.items()], ignore_index=True)
all_df.to_csv(shap_dir + "/importance_no_medication_with_group-nogridsearch.csv", index=False)

### Get top7 feature-sets per model

In [24]:
per_model_top_sets, group_to_cols = search_candidates(
    X_train, y_train, feature_cols, group_defs, models,
    shap_results, medication_features, include_med, 
    cv=5, use_smote=False, top_sets=7, max_k=10, verbose=True
)

XGBoost: 100%|██████████| 1023/1023 [07:47<00:00,  2.19it/s]


model: XGBoost, top10: ['PLT', 'Comorbidity', 'eGFR1', 'Age', 'PCT1', 'ALB', 'CRP1', 'D-d', 'BMI', 'Cr_baseline']
(['PLT', 'Comorbidity', 'eGFR1', 'Age', 'PCT1', 'ALB', 'CRP1', 'D-d', 'Cr_baseline'], 0.7693255295429208, 28)
(['PLT', 'Comorbidity', 'eGFR1', 'Age', 'PCT1', 'ALB', 'CRP1', 'D-d'], 0.7693199554069119, 27)
(['PLT', 'Comorbidity', 'Age', 'ALB', 'CRP1', 'BMI', 'Cr_baseline'], 0.7678595317725753, 26)
(['PLT', 'Comorbidity', 'Age', 'ALB', 'CRP1', 'D-d', 'Cr_baseline'], 0.7677591973244148, 26)
(['PLT', 'Comorbidity', 'eGFR1', 'Age', 'PCT1', 'ALB', 'CRP1', 'Cr_baseline'], 0.7669342251950948, 27)
(['PLT', 'Comorbidity', 'Age', 'ALB', 'CRP1', 'Cr_baseline'], 0.7665440356744705, 25)
(['PLT', 'Comorbidity', 'eGFR1', 'Age', 'ALB', 'CRP1', 'Cr_baseline'], 0.7659364548494982, 26)


LightGBM: 100%|██████████| 1023/1023 [03:01<00:00,  5.64it/s]


model: LightGBM, top10: ['PLT', 'Comorbidity', 'eGFR1', 'Age', 'PCT1', 'ALB', 'CRP1', 'D-d', 'Oxygen_concentration', 'WBC']
(['PLT', 'Comorbidity', 'eGFR1', 'Age', 'ALB', 'CRP1', 'Oxygen_concentration'], 0.7735117056856188, 26)
(['PLT', 'Comorbidity', 'eGFR1', 'Age', 'PCT1', 'ALB', 'CRP1', 'D-d'], 0.7731661092530657, 27)
(['PLT', 'Comorbidity', 'eGFR1', 'Age', 'PCT1', 'ALB', 'CRP1', 'D-d', 'Oxygen_concentration'], 0.7730602006688964, 28)
(['PLT', 'Comorbidity', 'eGFR1', 'Age', 'ALB', 'CRP1', 'D-d'], 0.7726532887402453, 26)
(['PLT', 'Comorbidity', 'eGFR1', 'Age', 'PCT1', 'ALB', 'CRP1'], 0.769046822742475, 26)
(['PLT', 'Comorbidity', 'eGFR1', 'Age', 'PCT1', 'ALB', 'CRP1', 'Oxygen_concentration'], 0.7683779264214048, 27)
(['PLT', 'Comorbidity', 'eGFR1', 'Age', 'ALB', 'CRP1', 'D-d', 'Oxygen_concentration'], 0.768227424749164, 27)


CatBoost: 100%|██████████| 1023/1023 [2:04:20<00:00,  7.29s/it] 


...

model: LogisticRegression, top10: ['Comorbidity', 'PLT', 'Immunosuppression', 'Age', 'N_percent', 'resistance_TGC', 'Oxygen_concentration', 'Resp_support', 'AST', 'ALB']
(['Comorbidity', 'PLT', 'Age', 'N_percent', 'resistance_TGC', 'Oxygen_concentration', 'AST', 'ALB'], 0.7732943143812709, 27)
(['Comorbidity', 'PLT', 'Age', 'N_percent', 'resistance_TGC', 'Oxygen_concentration', 'Resp_support', 'AST', 'ALB'], 0.7725306577480491, 28)
(['Comorbidity', 'PLT', 'Immunosuppression', 'Age', 'N_percent', 'resistance_TGC', 'Oxygen_concentration', 'AST', 'ALB'], 0.7716722408026756, 32)
(['Comorbidity', 'PLT', 'Immunosuppression', 'Age', 'N_percent', 'resistance_TGC', 'Oxygen_concentration', 'Resp_support', 'AST', 'ALB'], 0.7707357859531772, 33)
(['Comorbidity', 'PLT', 'Age', 'N_percent', 'resistance_TGC', 'Oxygen_concentration', 'Resp_support', 'ALB'], 0.7633277591973245, 27)
(['Comorbidity', 'PLT', 'Age', 'N_percent', 'resistance_TGC', 'Oxygen_concentration', 'ALB'], 0.7630936454849498, 26)
(['C

SVM: 100%|██████████| 1023/1023 [03:30<00:00,  4.87it/s]


model: SVM, top10: ['Comorbidity', 'PLT', 'Age', 'N_percent', 'resistance_TGC', 'eGFR1', 'CRP1', 'Oxygen_concentration', 'Immunosuppression', 'Cr_baseline']
(['Comorbidity', 'PLT', 'Age', 'N_percent', 'resistance_TGC', 'Oxygen_concentration', 'Immunosuppression', 'Cr_baseline'], 0.7841360089186177, 31)
(['Comorbidity', 'PLT', 'Age', 'N_percent', 'resistance_TGC', 'CRP1', 'Oxygen_concentration', 'Immunosuppression', 'Cr_baseline'], 0.7836064659977704, 32)
(['Comorbidity', 'PLT', 'Age', 'N_percent', 'CRP1', 'Oxygen_concentration', 'Immunosuppression', 'Cr_baseline'], 0.7805741360089187, 31)
(['Comorbidity', 'PLT', 'Age', 'N_percent', 'resistance_TGC', 'CRP1', 'Oxygen_concentration', 'Cr_baseline'], 0.7787681159420291, 27)
(['Comorbidity', 'PLT', 'Age', 'N_percent', 'resistance_TGC', 'CRP1', 'Immunosuppression', 'Cr_baseline'], 0.7787569676700112, 31)
(['Comorbidity', 'PLT', 'Age', 'N_percent', 'resistance_TGC', 'Oxygen_concentration', 'Cr_baseline'], 0.7781382385730212, 26)
(['Comorbidit

KNN: 100%|██████████| 1023/1023 [16:48<00:00,  1.01it/s]

model: KNN, top10: ['Comorbidity', 'Immunosuppression', 'PLT', 'N_percent', 'Cr_baseline', 'eGFR1', 'resistance_MIN', 'Coinfection_Fungi', 'Age', 'CRP1']
(['PLT', 'eGFR1', 'CRP1'], 0.6930936454849499, 13)
(['Comorbidity', 'Immunosuppression', 'PLT', 'N_percent', 'Cr_baseline', 'eGFR1', 'Coinfection_Fungi'], 0.6889604236343366, 30)
(['PLT', 'Age', 'CRP1'], 0.6841722408026756, 13)
(['Comorbidity', 'PLT', 'N_percent', 'Cr_baseline', 'Age', 'CRP1'], 0.6835144927536232, 25)
(['PLT', 'eGFR1', 'Age', 'CRP1'], 0.6833500557413601, 14)
(['Comorbidity', 'Immunosuppression', 'PLT', 'N_percent', 'Cr_baseline', 'Age', 'CRP1'], 0.6794788182831661, 30)
(['Comorbidity', 'Immunosuppression', 'PLT', 'N_percent', 'Cr_baseline', 'eGFR1', 'Coinfection_Fungi', 'CRP1'], 0.6776114827201783, 31)


### Evaluate candidates(7*7)

In [25]:
candidates = union_dedup_candidates(per_model_top_sets)
set_scores_df, candidate_details = evaluate_candidates(candidates, models, X_train, y_train, group_to_cols, medication_features, include_med, cv=5, use_smote=False)

In [26]:
feature_selection_dir = SAVE_DIR +'/feature_selection'
os.makedirs(feature_selection_dir, exist_ok=True)
set_scores_df.to_csv(feature_selection_dir + "/set_scores_stageB-nogridsearch.csv", index=False, encoding="utf-8-sig")

In [27]:
for i in candidate_details:
    print(i, candidate_details[i])
rows = []
for set_id, d in candidate_details.items():
    groups_str = "|".join(d["groups"])
    for model_name, m in d["per_model_metrics"].items():
        row = {"set_id": set_id, "groups": groups_str, "model": model_name, **m}
        rows.append(row)
pd.DataFrame(rows).to_csv(feature_selection_dir + "/candidate_details-nogridsearch.csv", index=False, encoding="utf-8-sig")

cand_01 {'groups': ['PLT', 'Comorbidity', 'eGFR1', 'Age', 'PCT1', 'ALB', 'CRP1', 'D-d', 'Cr_baseline'], 'cols': ['colistin_cms_daily_freq', 'polymyxin_b_daily_freq', 'colistin_sulfate_daily_freq', 'carbapenem_daily_dose', 'sulbactam_daily_dose', 'tigecycline_daily_dose', 'minocycline_daily_dose', 'vancomycin_daily_dose', 'eravacycline_daily_dose', 'aminoglycoside_daily_dose', 'PLT', 'Diabetes Mellitus', 'Hypertension', 'Heart Disease', 'Stroke', 'Malignant Tumor', 'Chronic Kidney Disease', 'Chronic Liver Disease', 'COPD', 'Comorb_other', 'Comorb_count', 'eGFR1', 'Age', 'PCT1', 'ALB', 'CRP1', 'D-d', 'Cr_baseline'], 'per_model_metrics': {'XGBoost': {'Accuracy': 0.7493360572012258, 'Precision': 0.8174288333714186, 'Recall': 0.8492307692307692, 'F1-score': 0.8326306956382423, 'AUROC': 0.7693255295429208, 'AUPRC': 0.8981543941951806}, 'LightGBM': {'Accuracy': 0.7538304392236976, 'Precision': 0.8236363636363636, 'Recall': 0.8461538461538461, 'F1-score': 0.8346282160022618, 'AUROC': 0.7580211

### Select final feature-set by metric

In [56]:
best_set_per_metric = select_final_sets(set_scores_df, candidate_details, drop_pct=0.4, min_models_kept=6, verbose=True)
print("best_set_per_metric:", best_set_per_metric)
final_set_ids = list(dict.fromkeys(best_set_per_metric.values()))
print("final_set_ids:", final_set_ids)

best_set_per_metric: {'Accuracy': 'cand_37', 'Precision': 'cand_16', 'Recall': 'cand_37', 'F1-score': 'cand_37', 'AUROC': 'cand_16', 'AUPRC': 'cand_37'}
final_set_ids: ['cand_37', 'cand_16']


### 5-fold on final feature-set

In [57]:
final_set_eval, final_set_metrics = evaluate_final_sets(best_set_per_metric, candidate_details, models, X_train, y_train, cv=5, use_smote=False)
final_set_metrics.to_csv(SAVE_DIR + "/final_set_metrics-nogridsearch.csv", index=False)

set_id: cand_37, selected_by: ['Accuracy', 'Recall', 'F1-score', 'AUPRC']
============================== XGBoost ============================== 

Accuracy: 0.7719 | Precision: 0.8361 | Recall: 0.8585 | F1-score: 0.8469 | AUROC: 0.7780 | AUPRC: 0.9002 | 

============================== LightGBM ============================== 

Accuracy: 0.7630 | Precision: 0.8279 | Recall: 0.8554 | F1-score: 0.8413 | AUROC: 0.7670 | AUPRC: 0.8962 | 

============================== CatBoost ============================== 

Accuracy: 0.7675 | Precision: 0.8435 | Recall: 0.8400 | F1-score: 0.8415 | AUROC: 0.7905 | AUPRC: 0.9033 | 

============================== RandomForest ============================== 

Accuracy: 0.7562 | Precision: 0.7617 | Recall: 0.9723 | F1-score: 0.8541 | AUROC: 0.7677 | AUPRC: 0.8893 | 

============================== LogisticRegression ============================== 

Accuracy: 0.7065 | Precision: 0.8627 | Recall: 0.7138 | F1-score: 0.7810 | AUROC: 0.7510 | AUPRC: 0.8710 | 

===